<a href="https://colab.research.google.com/github/arpitelias/ArpitJoshuaElias_FlyRank/blob/main/work/notebooks/w07_action_playbook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/arpitelias/ArpitJoshuaElias_FlyRank/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [7]:
!pip install -q duckdb
import duckdb, pandas as pd, numpy as np, os, json
from google.colab import userdata
SEED = 42
con = duckdb.connect()
con.execute("INSTALL httpfs; LOAD httpfs;")
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{userdata.get('HF_TOKEN')}')")
rel = "hf://datasets/FlyRank/internship-warehouse"
mar = f"read_parquet('{rel}/fact_content_daily_performance/month=2026-03/data_0.parquet')"
apr_p = f"read_parquet('{rel}/fact_content_daily_performance/month=2026-04/data_0.parquet')"
content = f"read_parquet('{rel}/dim_content.parquet')"
os.makedirs("work/outputs", exist_ok=True)
os.makedirs("work/figures", exist_ok=True)
df = con.sql(f"""
SELECT d.content_hash_id, d.client_hash_id,
       SUM(d.gsc_clicks) AS clicks, SUM(d.gsc_impressions) AS impressions,
       SUM(d.gsc_sum_position) / NULLIF(SUM(d.gsc_impressions), 0) AS avg_position,
       COUNT(DISTINCT d.report_date) AS days_seen,
       ANY_VALUE(c.word_count) AS word_count, ANY_VALUE(c.search_volume) AS search_volume,
       ANY_VALUE(c.main_intent) AS main_intent, ANY_VALUE(c.content_type) AS content_type,
       ANY_VALUE(c.is_deleted) AS is_deleted
FROM {mar} d
LEFT JOIN {content} c ON d.content_hash_id = c.content_hash_id AND d.client_hash_id = c.client_hash_id
WHERE d.gsc_data_available IS TRUE
GROUP BY 1, 2 HAVING SUM(d.gsc_impressions) >= 100
""").df()
df["ctr"] = df["clicks"] / df["impressions"] * 100
apr = con.sql(f"""
SELECT content_hash_id, client_hash_id, SUM(gsc_clicks) AS clicks_apr, SUM(gsc_impressions) AS impressions_apr,
       SUM(gsc_sum_position) / NULLIF(SUM(gsc_impressions), 0) AS avg_position_apr
FROM {apr_p} WHERE gsc_data_available IS TRUE GROUP BY 1,2 HAVING SUM(gsc_impressions) >= 100
""").df()
apr["ctr_apr"] = apr["clicks_apr"] / apr["impressions_apr"] * 100
scope = df[(df["is_deleted"] != True) & (df["avg_position"] <= 20)].copy()
scope["pos_bucket"] = pd.cut(scope["avg_position"], [0, 3, 10, 20], labels=["1-3", "4-10", "11-20"])
peer = scope.groupby("pos_bucket", observed=True)["ctr"].median().rename("peer_ctr")
scope = scope.join(peer, on="pos_bucket")
scope["shortfall"] = (scope["peer_ctr"] - scope["ctr"]).clip(lower=0)
scope["score"] = scope["shortfall"] * scope["impressions"]
d = scope.merge(apr[["content_hash_id","client_hash_id","ctr_apr","impressions_apr","avg_position_apr"]],
                on=["content_hash_id","client_hash_id"], how="left")
d = d.sort_values(["client_hash_id","content_hash_id"]).reset_index(drop=True)
d["label"] = np.where(d["ctr_apr"].notna(), (d["ctr_apr"] < d["peer_ctr"]).astype(float), np.nan)
print(f"in scope (position 1-20, not deleted, >=100 impressions): {len(d):,} pages, {d['client_hash_id'].nunique()} clients")
print(f"excluded 21-50 bucket by design (peer median CTR is 0.000, no page there can be labelled)")
print(f"with an April outcome to check against: {d['label'].notna().sum():,}")
print(f"peer medians:\n{peer.round(3).to_string()}")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

in scope (position 1-20, not deleted, >=100 impressions): 77,523 pages, 43 clients
excluded 21-50 bucket by design (peer median CTR is 0.000, no page there can be labelled)
with an April outcome to check against: 68,476
peer medians:
pos_bucket
1-3      0.211
4-10     0.190
11-20    0.101


## 1. Ranked actions + reason codes

The queue. 22,177 pages of 77,523 in scope carry a recommendation. The rest are either already at or above the median for their position band, or too thin to judge.

Pages are ranked by CTR shortfall against the median of their own position band, multiplied by impressions. A page is only in the queue if it falls below its band's median, and the volume term decides the order, because closing a small gap on a page seen 200,000 times moves more clicks than closing a large one on a page seen 500 times.

Reason codes and what each one means a reviewer should do:

Reason code	Pages	What it means	Action
visible_but_unclicked	2,853	Ranks in the top 3 but gets 5 clicks or fewer	Read the title and description against the query the page targets. A page this visible with almost no clicks usually promises something the searcher did not want. Check intent match first, not wording.
high_volume_shortfall	6,694	Below its band median, in the top quartile for impressions	Rewrite title and meta description. Do these first: highest traffic, so a small CTR gain moves the most clicks.
mid_page_shortfall	12,630	Below its band median, ordinary traffic	Metadata refresh when capacity allows. Lower priority than the two above.
too_thin_to_judge	16,568	Under 500 impressions a month	No action. CTR is too noisy at that volume to act on. Monitor only.
performing_at_peer	38,778	At or above its band median	Not in the queue.

How well the order holds up. Checked against April CTR, an outcome the score never saw: P@10 0.90, P@20 0.95, P@50 0.92, P@100 0.92, against a base rate of 0.557. So of the 50 pages a reviewer would work through first, about 46 were still below their band median a month later.

Note the base rate is higher here than in my earlier notebooks (0.46), because pages ranked 21-50 are now out of scope. Those were all forced zeros by how the label was built, and carrying them made the queue look better than it was. The honest comparison is 0.92 against 0.557.

Scope. Positions 1-20 only, not deleted, at least 100 impressions in the month. 17,188 pages ranked 21-50 are excluded by design: the median CTR in that band is 0.000, so no page there can register a shortfall or carry a label. Scoring them would produce rows a reviewer could never act on.

In [8]:
def precision_at_k(s, y, k):
    m = y.notna()
    s2, y2 = np.asarray(s)[m.values], np.asarray(y)[m.values]
    o = np.argsort(-s2)
    return float(y2[o[:k]].mean())
hi_vol = d["impressions"].quantile(0.75)
def archetype(r):
    if r["shortfall"] <= 0:
        return "performing_at_peer"
    if r["impressions"] < 500:
        return "too_thin_to_judge"
    if r["avg_position"] <= 3 and r["clicks"] <= 5:
        return "visible_but_unclicked"
    if r["impressions"] >= hi_vol:
        return "high_volume_shortfall"
    return "mid_page_shortfall"
d["archetype"] = d.apply(archetype, axis=1)
ACTIONS = {
 "visible_but_unclicked": ("Read the title and description against the query the page targets. A page this visible with almost no clicks usually promises something the searcher did not want.", "check intent match first, not wording"),
 "high_volume_shortfall": ("Rewrite title and meta description. Highest traffic, so a small CTR gain moves the most clicks.", "do these first"),
 "mid_page_shortfall": ("Metadata refresh when capacity allows.", "lower priority than the two above"),
 "too_thin_to_judge": ("No action. Under 500 impressions a month, CTR is too noisy to act on.", "monitor only"),
 "performing_at_peer": ("No action. At or above the median for its position band.", "not in the queue"),
}
d["action"] = d["archetype"].map(lambda a: ACTIONS[a][0])
d["reason_code"] = d["archetype"]
q = d[d["archetype"].isin(["visible_but_unclicked","high_volume_shortfall","mid_page_shortfall"])].copy()
q = q.sort_values("score", ascending=False).reset_index(drop=True)
q["rank"] = q.index + 1
print("archetype spread across the in-scope pages:")
print(d["archetype"].value_counts().to_string())
print(f"\nactionable queue: {len(q):,} pages of {len(d):,} in scope")
print(f"\nprecision@K of the queue against the April outcome (base rate {d['label'].mean():.3f}):")
for k in (10, 20, 50, 100):
    print(f"  P@{k:<4} {precision_at_k(q['score'], q['label'], k):.2f}")
print(f"\ntop 10:")
print(q.head(10)[["rank","archetype","impressions","clicks","ctr","peer_ctr","avg_position","score"]].round(2).to_string(index=False))


archetype spread across the in-scope pages:
archetype
performing_at_peer       38778
too_thin_to_judge        16568
mid_page_shortfall       12630
high_volume_shortfall     6694
visible_but_unclicked     2853

actionable queue: 22,177 pages of 77,523 in scope

precision@K of the queue against the April outcome (base rate 0.557):
  P@10   0.90
  P@20   0.95
  P@50   0.92
  P@100  0.92

top 10:
 rank             archetype  impressions  clicks  ctr  peer_ctr  avg_position    score
    1 high_volume_shortfall     212404.0    24.0 0.01      0.21          0.67 42505.71
    2 visible_but_unclicked     134984.0     1.0 0.00      0.21          2.69 28437.84
    3 visible_but_unclicked     124075.0     1.0 0.00      0.21          0.31 26131.50
    4 high_volume_shortfall     143019.0    43.0 0.03      0.19          3.17 22889.92
    5 high_volume_shortfall     107584.0    15.0 0.01      0.19          9.74 18953.23
    6 visible_but_unclicked      83834.0     1.0 0.00      0.21          0.12 1762

## 2. Intended use and limits

Who uses this. A content or SEO reviewer choosing which pages to look at this week. They open a page, read its title and meta description against what the page actually delivers, and rewrite if there is a mismatch. Cheap, reversible work.

What it is. A prioritisation aid. It says look here first. It does not say this page will improve, and it does not say what to write.

Where it stops being valid.

The queue is far longer than anyone will ever work through. 22,177 pages at 20 reviews a week is 21 years. Only the top of the list has any practical meaning, which is why I measure precision at 10, 20, 50 and 100 rather than over the whole ranking. Below a few hundred rows the order is untested.

The click arithmetic is arithmetic, not a forecast. If the top 200 pages each reached the median CTR of their position band, that would be about 12,262 more clicks a month on 9.2 million impressions. That is what the gap adds up to, not what rewriting would deliver. It assumes every page can reach its band median, that a rewrite achieves it, and that nothing else changes. I have no before-and-after measurement of a rewrite in this data, so I cannot put a number on the real effect.

It does not know why a page underperforms. The peer median treats every page in a position band as comparable, and I have no query intent in the score. A branded navigational page and an informational one at the same position are judged against the same median. A page whose low CTR is explained by its query type is a false positive, and nothing here can tell a reviewer which those are.

It is one month of inputs against one month of outcomes. No seasonality, no trend, no second window to confirm it.

Coverage varies a lot by client. Across 21 clients with 200 or more in-scope pages, the share flagged as actionable runs from 8% to 39%, median 19%. A client at 8% gets a short queue; one at 39% gets a list longer than they can use. The score does not adapt to that.

The top of the list is concentrated. 14 clients appear in the top 200 and the largest holds 74 of them, 37%. If the queue were worked strictly in order, one client would receive most of the attention. A reviewer working per client should rank within that client instead.

In [9]:
cap = 20
weeks_to_clear = len(q) / cap / 52
print(f"queue length {len(q):,} pages; at {cap} reviews a week that is {weeks_to_clear:.0f} years to clear")
print(f"top 200 covers {200/len(q):.1%} of the queue\n")
print("clicks currently captured vs the band median, top 200 by score:")
t = q.head(200)
gain = (t["peer_ctr"] - t["ctr"]) / 100 * t["impressions"]
print(f"  combined impressions: {t['impressions'].sum():,.0f}")
print(f"  clicks now: {t['clicks'].sum():,.0f}")
print(f"  clicks if each reached its band median: {(t['clicks'] + gain).sum():,.0f}")
print(f"  arithmetic difference: {gain.sum():,.0f} clicks a month (NOT a forecast, see limits)\n")
print("client concentration in the top 200:")
cc = t["client_hash_id"].value_counts()
print(f"  {len(cc)} clients, largest holds {cc.iloc[0]} ({cc.iloc[0]/200:.0%})")
print("\ncoverage: share of each client's in-scope pages that are actionable")
cov = d.groupby("client_hash_id").apply(lambda g: pd.Series({
    "in_scope": len(g), "actionable": g["archetype"].isin(["visible_but_unclicked","high_volume_shortfall","mid_page_shortfall"]).mean()}), include_groups=False)
cov = cov[cov["in_scope"] >= 200]
print(f"  across {len(cov)} clients with 200+ pages: {cov['actionable'].min():.0%} to {cov['actionable'].max():.0%}, median {cov['actionable'].median():.0%}")


queue length 22,177 pages; at 20 reviews a week that is 21 years to clear
top 200 covers 0.9% of the queue

clicks currently captured vs the band median, top 200 by score:
  combined impressions: 9,185,270
  clicks now: 5,667
  clicks if each reached its band median: 17,929
  arithmetic difference: 12,262 clicks a month (NOT a forecast, see limits)

client concentration in the top 200:
  14 clients, largest holds 74 (37%)

coverage: share of each client's in-scope pages that are actionable
  across 21 clients with 200+ pages: 8% to 39%, median 19%


## 3. Human review + the no-go list

What a person must check before acting on a row.

Does the page answer the query it ranks for? The score cannot see intent. If a page ranks for something it was never about, rewriting the title will not fix it and may make it worse.
Is this a branded or navigational query? Those get clicks or not for reasons unrelated to the title. The top 200 contains no navigational pages, so the problem is not visible here, but the score has no rule preventing it and the next month's queue might.
Is the current title actually wrong? 190 of the top 200 are flagged for traffic volume rather than anything observed about the page. The score has never read the title.
Is the page meant to get clicks at all? Some pages exist to be found, not clicked.
Was it changed recently? A page rewritten last week will look unchanged in this month's data.

What should never be automated.

Writing or publishing the new title and description. The whole point of the queue is that a human reads the page against the query. Automating the output removes the only step that checks whether the flag was right.

Acting on a single month. Nothing should trigger a change on one month's data without someone confirming the page looks the same way in a second window.

Any page with fewer than 500 impressions. 16,568 pages are excluded for this reason and none of them should be reintroduced by a threshold change without re-checking the noise.

Ranking across clients when the work is per client. One client holds 37% of the top 200. An automated run working strictly down the global list would spend most of its budget on one site.

Deciding a rewrite worked. The score cannot measure its own effect. Judging success needs a before-and-after comparison against similar pages that were not touched, which this data does not support.

One thing the numbers make obvious. The median page in the top 200 has 37,822 impressions against 1,599 across the queue, so the volume term dominates completely at the top. A reviewer should know they are being sent to the biggest pages with a gap, not the most broken ones.

In [10]:
t = q.head(200).copy()
print("what a reviewer is actually looking at, top 200:")
print(t["archetype"].value_counts().to_string())
print()
print("how extreme the top of the list is:")
print(f"  pages with zero clicks: {(t['clicks'] == 0).sum()}")
print(f"  pages with <= 5 clicks: {(t['clicks'] <= 5).sum()}")
print(f"  median impressions: {t['impressions'].median():,.0f} (whole queue: {q['impressions'].median():,.0f})")
print()
print("intent mix in the top 200 vs the queue (intent is NOT in the score):")
mix = pd.DataFrame({"top200": t["main_intent"].value_counts(normalize=True),
                    "queue": q["main_intent"].value_counts(normalize=True)}).round(3)
print(mix.to_string())
print()
print(f"main_intent missing: top 200 {t['main_intent'].isna().mean():.1%}, queue {q['main_intent'].isna().mean():.1%}")

what a reviewer is actually looking at, top 200:
archetype
high_volume_shortfall    190
visible_but_unclicked     10

how extreme the top of the list is:
  pages with zero clicks: 6
  pages with <= 5 clicks: 41
  median impressions: 37,822 (whole queue: 1,599)

intent mix in the top 200 vs the queue (intent is NOT in the score):
               top200  queue
main_intent                 
commercial      0.242  0.187
informational   0.596  0.581
navigational      NaN  0.002
transactional   0.162  0.230

main_intent missing: top 200 1.0%, queue 0.9%


## 4. Monitoring / retrain triggers

How long the queue stays valid. I rebuilt the same ranking on April data and compared it with the March one. Of the top 200, 84 pages are in both. So 58% of the list changes month to month. The queue has a shelf life of roughly a month, and a list more than a few weeks old is substantially wrong about what to look at first.

A fifth of the queue resolves on its own. Of the pages flagged in March, 78.7% were still below their band median in April, and 21.3% were not, with nobody having touched them. That is the honest denominator for any claim about this work: about one page in five would have looked fixed anyway. Any before-and-after measurement of a rewrite has to beat 21%, not 0%.

The rule's own thresholds move. The peer medians are recomputed from the data, and they shifted between months: the 1-3 band from 0.211 to 0.146, the 11-20 band from 0.101 to 0.064. A page can therefore change status without changing at all, because the bar moved. This is also a quiet warning about comparing scores across months: they are not on the same scale.

35.1% of pages changed position band between March and April, which is what drives most of the churn above.

Triggers, in the order I would check them.

Trigger	What to do
A month has passed	Rebuild the queue. It is cheap, and 58% of the top 200 will have changed.
A peer median moves more than 30% from the previous month	Stop and look before publishing a queue. Either the portfolio changed or the data did.
GSC availability drops below 36.7% of rows	Investigate before building anything. The label cannot exist without it, and a quiet collection failure would shrink the data without any error appearing.
Overlap of consecutive top 200s falls below 40%	Something structural has changed. At 42% overlap the current state is already near this line.
Reviewers report a false positive rate above 30%	The intent problem has become real. Stop and add a signal for query type.

What is not a trigger. A single page behaving oddly. The whole design assumes individual pages are noisy, which is why the volume floor and the position bands exist.

In [11]:
qa = q[q["label"].notna()].copy()
print("does the ranking still hold a month later?")
qa["score_apr"] = (qa["peer_ctr"] - qa["ctr_apr"]).clip(lower=0) * qa["impressions_apr"]
top_mar = set(qa.nlargest(200, "score")["content_hash_id"])
top_apr = set(qa.nlargest(200, "score_apr")["content_hash_id"])
print(f"  top 200 built on March vs rebuilt on April: {len(top_mar & top_apr)} of 200 pages in common")
print()
print("how much of the queue resolves without anyone acting:")
flagged = qa[qa["score"] > 0]
print(f"  flagged in March, still below band median in April: {flagged['label'].mean():.1%}")
print(f"  flagged in March, at or above band median in April: {1 - flagged['label'].mean():.1%}")
print()
print("inputs that would signal the recommendations have gone stale:")
print(f"  share of pages whose position band changed Mar to Apr: {(pd.cut(qa['avg_position_apr'], [0,3,10,20,999], labels=['1-3','4-10','11-20','21+']).astype(str) != qa['pos_bucket'].astype(str)).mean():.1%}")
apr_scope = apr[apr["avg_position_apr"] <= 20]
apr_peer = apr_scope.assign(b=pd.cut(apr_scope["avg_position_apr"], [0,3,10,20], labels=["1-3","4-10","11-20"])).groupby("b", observed=True)["ctr_apr"].median()
print("\n  peer medians, March vs April (the rule's own thresholds):")
print(pd.DataFrame({"march": peer.values, "april": apr_peer.values}, index=peer.index).round(3).to_string())

does the ranking still hold a month later?
  top 200 built on March vs rebuilt on April: 84 of 200 pages in common

how much of the queue resolves without anyone acting:
  flagged in March, still below band median in April: 78.7%
  flagged in March, at or above band median in April: 21.3%

inputs that would signal the recommendations have gone stale:
  share of pages whose position band changed Mar to Apr: 35.1%

  peer medians, March vs April (the rule's own thresholds):
            march  april
pos_bucket              
1-3         0.211  0.146
4-10        0.190  0.185
11-20       0.101  0.064


## 5. Exports for the paper

What the paper will build on.

File	What it is	Committed?
work/outputs/action_queue_top500.csv	The top 500 rows a reviewer would work through, with reason code, action and the numbers behind each	No, data file, regenerated by this notebook
work/outputs/action_queue_full.csv	All 22,177 queue rows	No, same reason, and 6 MB
work/outputs/playbook_metrics.json	Every number the paper quotes, with the scope line next to it	Yes
work/figures/precision_at_k.png	Queue precision at 10, 20, 50, 100 with the base rate marked	Yes
work/figures/archetypes.png	Pages by reason code across the in-scope set	Yes

The CSVs stay out of git by design: the repository's CI blocks data files, and this notebook regenerates them on any run. The metrics JSON is committed because it is the receipt the paper's numbers trace back to, and the figures are committed because the paper reuses them directly.

The precision figure carries the base rate line deliberately. A chart showing 0.92 with no base rate would overstate what the queue does; next to 0.557 it reads correctly, as roughly 1.65 times better than working through pages at random.

In [12]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
cols = ["rank","content_hash_id","client_hash_id","archetype","reason_code","action",
        "impressions","clicks","ctr","peer_ctr","shortfall","avg_position","score"]
q.head(500)[cols].to_csv("work/outputs/action_queue_top500.csv", index=False)
q[cols].to_csv("work/outputs/action_queue_full.csv", index=False)
fig, ax = plt.subplots(figsize=(6, 3.2), dpi=160)
ks = [10, 20, 50, 100]
ax.bar([str(k) for k in ks], [precision_at_k(q["score"], q["label"], k) for k in ks], color="#2D5F5D")
ax.axhline(d["label"].mean(), color="#6B6B6B", linestyle="--", linewidth=1)
ax.text(2.6, d["label"].mean() + 0.02, f"base rate {d['label'].mean():.3f}", color="#6B6B6B", fontsize=8)
ax.set_ylim(0, 1); ax.set_xlabel("top K of the queue"); ax.set_ylabel("precision")
ax.set_title("Queue precision against April outcome", fontsize=10)
fig.tight_layout(); fig.savefig("work/figures/precision_at_k.png"); plt.close(fig)
fig, ax = plt.subplots(figsize=(6, 3.2), dpi=160)
counts = d["archetype"].value_counts()
ax.barh(counts.index[::-1], counts.values[::-1], color="#2D5F5D")
ax.set_xlabel("pages"); ax.set_title("Pages by reason code, positions 1-20", fontsize=10)
fig.tight_layout(); fig.savefig("work/figures/archetypes.png"); plt.close(fig)
metrics = {
 "scope": "2026-03 features, 2026-04 outcome; positions 1-20; >=100 impressions; not deleted",
 "excluded": "position 21-50 band (peer median CTR 0.000, no page can be labelled)",
 "pages_in_scope": int(len(d)), "pages_with_outcome": int(d["label"].notna().sum()),
 "queue_length": int(len(q)), "base_rate": round(float(d["label"].mean()), 4),
 "precision_at": {str(k): round(precision_at_k(q["score"], q["label"], k), 4) for k in ks},
 "peer_medians_march": {str(k): round(float(v), 4) for k, v in peer.items()},
 "top200_overlap_march_april": 84,
 "share_resolving_unaided": 0.213,
 "queue_clears_in_years_at_20_per_week": round(len(q) / 20 / 52, 1),
 "seed": SEED,
}
with open("work/outputs/playbook_metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)
print(json.dumps(metrics, indent=2))
print("\nwritten:")
for p in ["work/outputs/action_queue_top500.csv","work/outputs/action_queue_full.csv",
          "work/outputs/playbook_metrics.json","work/figures/precision_at_k.png","work/figures/archetypes.png"]:
    print(f"  {p}  {os.path.getsize(p):,} bytes")


{
  "scope": "2026-03 features, 2026-04 outcome; positions 1-20; >=100 impressions; not deleted",
  "excluded": "position 21-50 band (peer median CTR 0.000, no page can be labelled)",
  "pages_in_scope": 77523,
  "pages_with_outcome": 68476,
  "queue_length": 22177,
  "base_rate": 0.5566,
  "precision_at": {
    "10": 0.9,
    "20": 0.95,
    "50": 0.92,
    "100": 0.92
  },
  "peer_medians_march": {
    "1-3": 0.2114,
    "4-10": 0.1901,
    "11-20": 0.1008
  },
  "top200_overlap_march_april": 84,
  "share_resolving_unaided": 0.213,
  "queue_clears_in_years_at_20_per_week": 21.3,
  "seed": 42
}

written:
  work/outputs/action_queue_top500.csv  153,664 bytes
  work/outputs/action_queue_full.csv  5,983,757 bytes
  work/outputs/playbook_metrics.json  602 bytes
  work/figures/precision_at_k.png  28,069 bytes
  work/figures/archetypes.png  36,366 bytes


## Self-check

Before you submit, confirm each line honestly:

- [ ✅] Every section above is filled — markdown thinking AND the code that backs it
- [✅ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [✅ ] No client names, URLs, or private queries anywhere
- [✅ ] My claims use careful words: observed, measured, directional, decision-support
- [✅ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.